# Phase 2.5: Price Level Features (Budget Input)

Extract price levels from tips and integrate as a feature for budget-aware recommendations.

This allows users to filter by budget: Low / Medium / High

In [ ]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import numpy as np
from config import *
from src.data.loader import load_businesses, load_tips
from src.data.cache import DataCache
from src.features.price_level_features import PriceLevelFeatureBuilder

cache = DataCache(str(CACHE_DIR))

# Load cached data from earlier phases
businesses = cache.load_dataframe('businesses_major_cities')
print(f"Loaded {len(businesses):,} businesses")

## Load Tips Data

In [ ]:
# Load tips (10% sample same as other data)
tips = load_tips(str(TIPS_FILE), sample=True, sample_fraction=PHASE1_SAMPLE_FRACTION)
print(f"Loaded {len(tips):,} tips")
print(f"\nTips columns: {tips.columns.tolist()}")
print(f"\nSample tips with price signals:")
print(tips[tips['text'].str.contains('price|cheap|expensive|deal', case=False, na=False)].head(3)[['business_id', 'text']])

## Build Price Level Features

In [ ]:
# Initialize builder
price_builder = PriceLevelFeatureBuilder()

# Build price levels for all businesses
print("Building price level features...")
price_features = price_builder.build_price_features(businesses, tips)

print(f"\n✓ Generated price levels for {len(price_features):,} businesses")
print(f"\nPrice level distribution:")
print(price_features['price_level'].value_counts().sort_index())
print(f"\nSource of price data:")
print(price_features['price_source'].value_counts())

## Merge with Business Data

In [ ]:
# Merge price levels back to businesses
businesses_with_price = businesses.merge(
    price_features[['business_id', 'price_level', 'price_source']], 
    on='business_id',
    how='left'
)

# Add human-readable labels
price_labels = {1: 'Low', 2: 'Medium', 3: 'High'}
businesses_with_price['price_label'] = businesses_with_price['price_level'].map(price_labels)

print(f"✓ Merged price levels with {len(businesses_with_price):,} businesses")
print(f"\nSample:")
print(businesses_with_price[['business_id', 'name', 'city', 'price_level', 'price_label', 'price_source']].head(10))

## Cache Price Features

In [ ]:
# Save the enhanced business data
cache.save_dataframe(businesses_with_price, 'businesses_with_price')
cache.save_dataframe(price_features, 'price_levels')

print("✓ Price features cached")
print("\nReady for Phase 3: Feature Engineering with Budget")

## Example: Budget-Filtered Recommendations

When user specifies a budget, we can filter recommendations:

In [ ]:
# Example: User has LOW budget
user_budget = 'Low'  # Could also be 'Medium' or 'High'

budget_filtered = businesses_with_price[businesses_with_price['price_label'] == user_budget]
print(f"Budget: {user_budget}")
print(f"Available restaurants: {len(budget_filtered):,}")
print(f"\nTop 5 by review count:")
print(budget_filtered.nlargest(5, 'review_count')[['name', 'city', 'stars', 'review_count', 'price_label']])

In [ ]:
# Distribution comparison: price level vs average stars
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Price level distribution
price_dist = businesses_with_price['price_label'].value_counts()
axes[0].bar(price_dist.index, price_dist.values, color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[0].set_title('Price Level Distribution')
axes[0].set_ylabel('Number of Restaurants')
axes[0].set_xlabel('Price Level')

# Average stars by price level
avg_stars = businesses_with_price.groupby('price_label')['stars'].mean()
axes[1].bar(avg_stars.index, avg_stars.values, color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1].set_title('Average Rating by Price Level')
axes[1].set_ylabel('Average Stars')
axes[1].set_xlabel('Price Level')
axes[1].set_ylim([0, 5])

plt.tight_layout()
plt.show()

print(f"\nAverage rating by price level:")
print(avg_stars)